# Register data types

A qubit can carry a computational basis value or a superposition. This guide maps every cell of the locked data-type tutorial onto FlagQuantum's existing owners. Register metadata belongs to Core, program construction to Compiler, numerical codecs to Simulation, and diagrams to Drawer. The first program prepares one zero qubit and one plus qubit.

In [ ]:
# Copyright 2024 Google LLC
# SPDX-License-Identifier: Apache-2.0
# Adapted from Qualtran v0.7.0 DataTypes.ipynb, revision
# bcb17bb25360ed68ce3bc084645db3e929799389.
from collections import Counter
from collections.abc import Hashable
from dataclasses import dataclass

import flagquantum as fq
from flagquantum.compiler.registers.component_builder import ComponentProgramBuilder
from flagquantum.compiler.registers.component_gate_function import bit_gate_function
from flagquantum.compiler.registers.register_binding import bind_register_program
from flagquantum.core.ir import CircuitIR, Instruction
from flagquantum.core.ir.program import register_types as types
from flagquantum.core.ir.program.register_bindings import RegisterBinding
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.component_score import component_register_score


@dataclass(frozen=True)
class RegisterOperation:
    ir: CircuitIR

    def append_to(self, circuit: fq.Circuit) -> fq.Circuit:
        return circuit.compose(self.ir)


bit = types.quantum_bit_type()
h_gate = bit_gate_function(
    "h", (Register("q", bit),), CircuitIR(1, (Instruction("h", (0,)),))
)
builder = ComponentProgramBuilder(
    Signature(()), (h_gate,), allow_register_addition=True
)
zero = builder.allocate(dtype=bit)
plus = builder.allocate(dtype=bit)
plus_result = builder.call_dict("h", {"q": plus})["q"]
assert isinstance(plus_result, RegisterBinding)
plus = plus_result
prepared = builder.finalize({"q0": zero, "q1": plus})
prepared_bound = bind_register_program(
    prepared, (), clean=(0, 1), allow_workspace_outputs=True
)
prepared_score = component_register_score(prepared)
prepared_circuit = fq.Circuit(2)
prepared_circuit.append(RegisterOperation(prepared_bound.circuit))
prepared_circuit.state()


## Owned quantum values

The builder returns an owned handle for every allocation and call. A call consumes its input handle and returns a successor. Finalization returns the remaining values. This linear ownership tracks logical dependencies separately from physical wire placement.

## Signatures

A Signature contains ordered Register declarations. A declaration records its name, dtype, shape and direction. The default direction is THRU. A signature can describe an operation before its implementation exists.

In [ ]:
signature = Signature((Register("arg1", bit), Register("arg2", bit)))
print(signature.n_qubits())


## Typed quantum words

Unsigned, signed, fixed-point and field metadata describe the interpretation of a register. A word is one logical value with a declared width. Declaring two 32-bit inputs needs two declarations. Numerical execution and physical decomposition apply their own admission rules.

In [ ]:
signature = Signature(
    (
        Register("x", types.unsigned_register_type(32)),
        Register("y", types.unsigned_register_type(32)),
    )
)
print(signature.n_qubits())


### Compile-time type parameters

Large words can be described with compact signatures and resource-call graphs. The following 2048-bit negation graph contains BitwiseNot and AddK(1). Its construction is metadata work. Physical simulation still scales with the circuit and state representation.

In [ ]:
from flagquantum.compiler._component_trace.arithmetic_call import ArithmeticCall
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph

large_signature = Signature((Register("x", types.unsigned_register_type(2048)),))
large_calls: dict[Hashable, int] = {
    ArithmeticCall("bitwise_not", (2048,), ("unsigned",)): 1,
    ArithmeticCall("constant_add", (2048,), ("unsigned",), constant=1): 1,
}
large_root: Hashable = "Negate"
large_declarations: dict[Hashable, dict[Hashable, int]] = {large_root: large_calls}
large_graph = symbolic_call_graph(large_root, large_declarations, max_depth=1)
large_signature.n_qubits(), large_graph


## Splitting and joining

A decomposition can split a word into ordered bit handles, apply elementary gates and rejoin the successor handles with the original dtype. The following complete three-bit NOT program uses one shared X definition and supports circuit.append(op).

In [ ]:
dtype = types.unsigned_register_type(3)
x_gate = bit_gate_function(
    "x", (Register("q", bit),), CircuitIR(1, (Instruction("x", (0,)),))
)
builder = ComponentProgramBuilder(Signature((Register("x", dtype),)), (x_gate,))
x_word = builder.inputs["x"]
assert isinstance(x_word, RegisterBinding)
x_bits = builder.split(x_word)
for index in range(len(x_bits)):
    x_bits[index] = builder.call_dict("x", {"q": x_bits[index]})["q"]
x_word = builder.join(x_bits, dtype=dtype)
complement = builder.finalize({"x": x_word})
complement_bound = bind_register_program(complement, ((0, 1, 2),))
complement_score = component_register_score(complement)
circuit = fq.Circuit(3)
circuit.append(RegisterOperation(complement_bound.circuit))
observed_counts = dict(Counter(gate.name for gate in circuit.to_ir().instructions))
assert observed_counts == {"x": 3}
complement_score


## Bit order

Bit index zero is the most significant bit. The integer 0x30 therefore encodes as 0, 0, 1, 1, 0, 0, 0, 0 in an eight-bit word.

In [ ]:
from flagquantum.simulation.integer_encoding import integer_to_bits

integer_to_bits(0x30, 8)


## Storage casts and opaque words

An opaque word retains a width without asserting a numerical interpretation. Joining bits without a dtype creates that opaque word. A same-width cast changes its interpretation while preserving wire order. The next program allocates three zero bits, joins them and casts the result to unsigned.

In [ ]:
from flagquantum.compiler.registers.layout.component_cast import cast_word_function

builder = ComponentProgramBuilder(Signature(()), allow_register_addition=True)
zeros = [builder.allocate(dtype=bit) for _ in range(3)]
opaque_word = builder.join(zeros)
cast = cast_word_function("as_unsigned", types.opaque_register_type(3), dtype)
builder.define(cast)
x_word = builder.call_dict(cast.name, {"reg": opaque_word})["reg"]
cast_program = builder.finalize({"x": x_word})
cast_bound = bind_register_program(
    cast_program, (), clean=(0, 1, 2), allow_workspace_outputs=True
)
assert not cast_bound.circuit.instructions
component_register_score(cast_program)


## Type compatibility

Equal declarations and single-bit quantum types are compatible. STRICT excludes the additional loose rules. LOOSE also admits same-width opaque words, compatible integer encodings and unsigned integer-only fixed-point declarations. Construction, value validation and physical admission remain separate checks.

In [ ]:
from flagquantum.core.ir.program.register_compatibility import (
    RegisterTypeCheckingSeverity,
    check_register_types_consistent,
)

unsigned = types.unsigned_register_type(3)
opaque = types.opaque_register_type(3)
strict = RegisterTypeCheckingSeverity.STRICT
compatibility = (
    check_register_types_consistent(unsigned, unsigned),
    check_register_types_consistent(bit, types.opaque_register_type(1)),
    check_register_types_consistent(opaque, unsigned),
    check_register_types_consistent(opaque, unsigned, strict),
    check_register_types_consistent(unsigned, types.signed_register_type(3)),
    check_register_types_consistent(unsigned, types.signed_register_type(3), strict),
    check_register_types_consistent(opaque, types.opaque_register_type(4)),
)
assert compatibility == (True, True, True, False, True, False, False)
compatibility


## Quantum and classical storage

RegisterType stores quantum and classical widths directly. Existing constructors retain each encoding's parameters. Shared numerical codecs handle the value domain and bits. The table below includes both storage categories. Wider classical declarations can be inspected and encoded here; their Program execution extension remains pending Proposal 108.

In [ ]:
from flagquantum.simulation.finite_field_encoding import FiniteFieldEncoding
from flagquantum.simulation.finite_field_polynomial_encoding import (
    FiniteFieldPolynomialEncoding,
)

field = FiniteFieldEncoding(3, 4)
polynomial = FiniteFieldPolynomialEncoding(4, FiniteFieldEncoding(2, 2))
dtypes = (
    bit,
    types.classical_bit_type(),
    types.opaque_register_type(8),
    types.signed_register_type(8),
    types.classical_signed_register_type(8),
    types.ones_complement_register_type(8),
    types.classical_ones_complement_register_type(8),
    types.unsigned_register_type(8),
    types.classical_unsigned_register_type(8),
    types.bounded_unsigned_register_type(8, 230),
    types.classical_bounded_unsigned_register_type(8, 230),
    types.fixed_point_register_type(8, 2),
    types.classical_fixed_point_register_type(8, 2),
    types.montgomery_register_type(8, 227),
    types.classical_montgomery_register_type(8, 227),
    field.register_type(),
    field.register_type(classical=True),
    polynomial.register_type(),
    polynomial.register_type(classical=True),
)


### Declaration properties

Each row shows the encoding and physical storage widths. These native properties replace foreign dtype inheritance checks. Display labels are native encoding names.

In [ ]:
properties = tuple(
    (dtype.encoding, dtype.num_bits, dtype.num_qubits, dtype.num_cbits)
    for dtype in dtypes
)
print("encoding                 bits qubits cbits")
for encoding, width, quantum_width, classical_width in properties:
    print(f"{encoding:24} {width:4} {quantum_width:6} {classical_width:5}")


### Encoding examples

For each numerical declaration, take the first and last value in its original domain and encode them through the shared scalar codec. Opaque words have an ambiguous numerical domain. Fixed-point values here are stored integers. Field and polynomial values retain their existing numerical objects.

In [ ]:
from flagquantum.simulation.classical_domains import register_classical_domain
from flagquantum.simulation.classical_register_codec import encode_register_value

encoding_rows: list[tuple[list[int], ...] | None] = []
for dtype in dtypes:
    if dtype.encoding == "opaque":
        encoding_rows.append(None)
        print("opaque: ambiguous numerical domain")
        continue
    register = Register("value", dtype)
    values = list(register_classical_domain(register))
    endpoints = (values[0], values[-1])
    encoded = tuple(encode_register_value(register, value) for value in endpoints)
    encoding_rows.append(encoded)
    print(dtype.encoding, tuple(zip(endpoints, encoded, strict=True)))


## Adding a numerical encoding

Keep encoding arithmetic in Simulation and neutral declaration metadata in Core. Reuse scalar codecs through encode_value_array and decode_value_array when array behavior is required. An additional protected dtype or Program behavior requires the existing API proposal process. Custom value callbacks can already be composed without introducing a parallel dtype hierarchy.

This notebook retains the locked tutorial's 24-cell correspondence. Its 2048-bit graph is a resource declaration. The three-bit program below is the executable example, with emitted primitives counted separately.